# Link prediction on the F1 KGs: data checks and baselines

Builds the link-prediction graphs of all three KGs and checks them before any training
(`link_prediction_rgcn.ipynb` / `link_prediction_graphsage.ipynb` build the same data themselves):

| Target | Query | Fact | Candidates |
|---|---|---|---|
| Team | `RaceEntry → ?` | `hasTeam` | 182 Teams |
| Position | `RaceEntry → ?` | `hasStartPosition` (grid slot) | 34 Positions |
| Circuit | `Race → ?` | `heldAtCircuit` (Hybrid: `isInCircuit`) | 78 Circuits |

1. sizes and the per-race split,
2. every KG holds the same target facts,
3. no held-out fact is reachable in the message-passing graph, in any KG,
4. the start/finish clash of Pure SULO is gone (no finish position in any graph),
5. simple baselines, reported next to the GNNs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

from f1_node_data import load_population
from f1_population import SCHEMA_FILES, read_schema
from link_prediction.link_prediction_tools import (
    TARGETS, baseline_hits_at_1, build_lp_data, frequency_baseline, leaked_facts, race_split, target_facts,
)

KGS = ["adhoc", "pure_sulo", "hybrid_sulo"]
OUT_DIR = ROOT / "outputs" / "f1" / "link_prediction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data and split

`build_lp_data` renders each KG from `data/f1/assertions.csv` **without** `hasFinishPosition`, for
every `RaceEntry`: in Pure SULO the finish and start position are the same
`sulo:hasFeature → Position/<n>` triple whenever they are equal (1,976 entries), so a held-out grid
slot would otherwise stay in the graph. Every other outcome (points, status, laps) stays.

Holdout is per race (80/10/10 of the 1,153 races, `random_state=42`, shared by all KGs and both
encoders): a val/test race loses the facts of all three targets from message passing, along their
whole path (Pure's team: the `RaceEntry/<id>/hasTeam` Role node and its edges). A `hasTeamMember`
pair (Hybrid/Pure) is shown only while one of the entries behind it shows its team. During
training, each epoch hides and supervises a random `DISJOINT_TRAIN_RATIO` share of the training
races, so training queries look like test queries. Node features are those of node prediction;
the degree feature is recomputed on each visible graph.

In [ ]:
entities, assertions = load_population(ROOT / "data" / "f1")
split = race_split(entities, seed=42)
schemas = {kg: read_schema(str(ROOT / SCHEMA_FILES[kg])) for kg in KGS}
lp_data = {kg: build_lp_data(kg, entities, assertions, split, schemas[kg]) for kg in KGS}

pd.DataFrame({
    kg: {
        "nodes": d.data.num_nodes,
        "edges (fwd + inv)": d.data.edge_index.size(1),
        "relation types (fwd + inv)": d.num_relations,
        "races train / val / test": " / ".join(str(len(p)) for p in split),
        **{f"{name} facts train / val / test": " / ".join(map(str, torch.bincount(d.race_split[t.race], minlength=3).tolist()))
           for name, t in d.targets.items()},
        **{f"{name} path edges per fact": t.edge_fact.size(1) // len(t.head) for name, t in d.targets.items()},
        "hasTeamMember pairs": d.member_edges.size(1) // 2,
    }
    for kg, d in lp_data.items()
}).T

## 2. The same facts in every KG

Each target is read from the assertions in the property the KG's schema renders (Ad-hoc and Pure
`heldAtCircuit`, Hybrid `isInCircuit`), so the (head, answer) pairs must be identical.

In [ ]:
pairs = {kg: {t: set(zip(f["subject"], f["value"])) for t, f in target_facts(assertions, schemas[kg]).items()}
         for kg in KGS}
assert pairs["adhoc"] == pairs["pure_sulo"] == pairs["hybrid_sulo"]
{t: len(p) for t, p in pairs["adhoc"].items()}

## 3. No held-out fact is reachable

For both holdout stages (validation: val + test races hidden; test: test races hidden), every
hidden fact's head and answer must be non-adjacent in the visible graph, and its SULO intermediate
node (Pure's `RaceEntry/<id>/hasTeam` Role) must have no edge left. `team_via_member` counts hidden
team facts whose answer is still two hops away through the entry's driver
(`entry → Driver → hasTeamMember → Team`) although no visible entry of that driver shows that team.
`leaked_facts` checks all of this from the graph itself, independently of how the facts' edges were found.

In [ ]:
leaks = {(kg, stage): leaked_facts(d, d.race_split >= s) for kg, d in lp_data.items()
         for stage, s in (("val", 1), ("test", 2))}
leaks = pd.DataFrame(leaks).T
assert (leaks == 0).all().all(), leaks
leaks

## 4. No finish position in any graph

In Pure SULO, start and finish position are both `sulo:hasFeature → Position/<n>`, one merged triple
when they are equal. With `hasFinishPosition` removed, the only RaceEntry → Position edges left are
the grid facts, in every KG.

In [ ]:
value = lambda prop: assertions[assertions["property"] == prop].set_index("subject")["value"]
start, finish = value("hasStartPosition"), value("hasFinishPosition")
print("entries whose start and finish position are equal:", int((start == finish.reindex(start.index)).sum()))

for kg, d in lp_data.items():
    entries = torch.zeros(d.data.num_nodes, dtype=torch.bool)
    entries[d.targets["team"].head] = True          # every RaceEntry has a team
    positions = torch.zeros(d.data.num_nodes, dtype=torch.bool)
    positions[d.targets["position"].candidates] = True
    src, dst = d.data.edge_index
    to_position = int((entries[src] & positions[dst]).sum())
    assert to_position == len(d.targets["position"].head), kg
    print(f"{kg}: {to_position} RaceEntry -> Position edges = {len(d.targets['position'].head)} grid facts")

## 5. Baselines

Both see the train + val races only and are scored on the test races, like the final models:
- **frequency**: rank the candidates by how often they answered a train/val fact (MRR, Hits@1);
  the shuffled-answer canary's chance level;
- **one rule** (Hits@1): Team = the driver's most frequent team in that season (else over all
  seasons); Circuit = the race name's most frequent circuit; Position = the qualifying position,
  else the driver's most frequent grid slot that season (else over all seasons). Each falls back to
  the overall most frequent answer.

In [ ]:
frequency = frequency_baseline(lp_data["adhoc"], "test")   # identical facts, so identical in every KG
rules = baseline_hits_at_1(assertions, split)
baselines = pd.DataFrame({t: {"test queries": frequency[t]["num_queries"], "frequency MRR": frequency[t]["mrr"],
                              "frequency Hits@1": frequency[t]["hits@1"], "one-rule Hits@1": rules[t]}
                          for t in TARGETS}).T
baselines.to_csv(OUT_DIR / "baselines.csv")
baselines.round(4)